# Calibrate the Birth Rate and Validate the Model Growth Rate

In [1]:
from collections import defaultdict
from matplotlib.colors import LogNorm
from pathlib import Path
import jenkspy
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import shutil
import string

from country_config import *
from files_config import *
from run_config import *
from utils import *

## Calibrate Sim Birth Rate

Using 1 location and a fixed population of 200k, adjust birth rate to match target growth rate from Excel sheet real world data 

### Copy and setup files in _pre_sim_200k_pop_run dir

In [2]:
os.makedirs(pre_sim_run_path, exist_ok=True)
os.makedirs(PRE_SIM_RUN_PATH_INPUTS_DIR, exist_ok=True)

pop_calibration_config = {
    "betas": [0.0],
    "treatment_access_rates": [0.0],
    "population_bins": [200000],
}

for filename in os.listdir(PRE_SIM_RUN_PATH_INPUTS_DIR):
    if filename.lower().endswith((".yml", ".yaml")):
        os.remove(os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, filename))

#copy bin folder from template to calibration path
template_bin = os.path.join(template_path, "bin")
pre_calibration_bin = os.path.join(pre_sim_run_path, "bin")

if os.path.exists(template_bin):
    if os.path.exists(pre_calibration_bin):
        info(f"Info: bin folder already exists in pre sim run path: {pre_calibration_bin}. Skipping copy.")
    else:
        shutil.copytree(template_bin, pre_calibration_bin, copy_function=shutil.copy2, symlinks=True)
        print(f"Copied bin folder from {template_bin} to {pre_calibration_bin}")

→ Info: bin folder already exists in pre sim run path: _pre_sim_200k_pop_run/bin. Skipping copy.


In [3]:
# Template files for rasters (replace path if needed)
BETA_RASTER_TEMPLATE = f"{template_path}/{country_code}_beta.template"
ACCESS_RASTER_TEMPLATE = f"{template_path}/{country_code}_treatmentseeking.template"
POPULATION_BIN_RASTER_TEMPLATE = f"{template_path}/{country_code}_initialpopulation_1_location.template"

# Read templates
if not os.path.exists(TEMPLATE):
    raise FileNotFoundError(f"Input template not found: {TEMPLATE}")

with open(TEMPLATE, "r", encoding="utf-8") as f:
    input_template_text = f.read()

beta_raster_template_text = None
access_raster_template_text = None
population_bin_raster_template_text = None

if os.path.exists(BETA_RASTER_TEMPLATE):
    with open(BETA_RASTER_TEMPLATE, "r", encoding="utf-8") as f:
        beta_raster_template_text = f.read()
else:
    warn(f"Warning: beta raster template not found: {BETA_RASTER_TEMPLATE}. Skipping raster generation for beta files.")
    
if os.path.exists(ACCESS_RASTER_TEMPLATE):
    with open(ACCESS_RASTER_TEMPLATE, "r", encoding="utf-8") as f:
        access_raster_template_text = f.read()
else:
    warn(f"Warning: access raster template not found: {ACCESS_RASTER_TEMPLATE}. Skipping raster generation for access files.")

if os.path.exists(POPULATION_BIN_RASTER_TEMPLATE):
    with open(POPULATION_BIN_RASTER_TEMPLATE, "r", encoding="utf-8") as f:
        population_bin_raster_template_text = f.read()
else:
    warn(f"Warning: population bin raster template not found: {POPULATION_BIN_RASTER_TEMPLATE}. Skipping raster generation for population bin files.")

In [4]:
# Generate files for each combination of beta and access rate
# This will only run once with 0 beta, 0 access rate, and 200k fixed population
created = []
for i, b in enumerate(pop_calibration_config["betas"], start=1):
    for j, a in enumerate(pop_calibration_config["treatment_access_rates"], start=1):
        for k, p in enumerate(pop_calibration_config["population_bins"], start=1):
            beta_str = f"{b:.4f}"
            access_str = f"{a:.3f}"
            birth_rate_str = f"{birth_rate:.4f}"
            population_bin_str = f"{p}"

            # input file: replace all tags
            out_text = input_template_text.replace("#POPULATION_SCALE#", f"{pre_calibration_sim_scale}")
            out_text = out_text.replace("#COUNTRY_CODE#", f"{country_code}")
            out_text = out_text.replace("#INITIAL_YEAR#", f"{initial_year}")
            out_text = out_text.replace("#CALIBRATION_YEAR#", f"{calibration_year}")
            out_text = out_text.replace("#POPULATION#", f"_{population_bin_str}")
            out_text = out_text.replace("#CELL_LEVEL_REPORTING#", "false")
            out_text = out_text.replace("#BIRTH_RATE#", f"{birth_rate_str}")
            out_text = out_text.replace("#BETA#", f"_{beta_str}").replace("#ACCESS_RATE#", f"_{access_str}")
            out_text = out_text.replace("#INPUT_PATH#", f"{PRE_SIM_RUN_PATH_INPUTS_DIR}")
            out_text = out_text.replace("district.asc", "districts.asc")
            # out_text = out_text.replace("district.asc", "district_seq1.asc")

            out_name = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, f"input_beta_{beta_str}_access_{access_str}_pop_{population_bin_str}.yml")
            with open(out_name, "w", encoding="utf-8") as outf:
                outf.write(out_text)
            created.append(out_name)

            # beta raster
            if beta_raster_template_text is not None:
                beta_raster_text = beta_raster_template_text.replace("#BETA#", beta_str)
                beta_raster_name = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, f"{country_code}_beta_{beta_str}.asc")
                with open(beta_raster_name, "w", encoding="utf-8") as outf:
                    outf.write(beta_raster_text)
                created.append(beta_raster_name)

            # access raster
            if access_raster_template_text is not None:
                access_raster_text = access_raster_template_text.replace("#ACCESS_RATE#", access_str)
                access_raster_name = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, f"{country_code}_treatment_seeking_normalized_{access_str}.asc")
                with open(access_raster_name, "w", encoding="utf-8") as outf:
                    outf.write(access_raster_text)
                created.append(access_raster_name)
                
            # population bin raster
            if population_bin_raster_template_text is not None:
                population_bin_raster_text = population_bin_raster_template_text.replace("#POPULATION#", population_bin_str)
                population_bin_raster_name = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, f"{country_code}_initialpopulation_{population_bin_str}.asc")
                with open(population_bin_raster_name, "w", encoding="utf-8") as outf:
                    outf.write(population_bin_raster_text)
                created.append(population_bin_raster_name)

# Summary
print(f"Generated {len(created)} files in {PRE_SIM_RUN_PATH_INPUTS_DIR}")
for p in created[:10]:
    print("  ", p)
if len(created) > 10:
    print("  ...")



Generated 4 files in _pre_sim_200k_pop_run/input
   _pre_sim_200k_pop_run/input/input_beta_0.0000_access_0.000_pop_200000.yml
   _pre_sim_200k_pop_run/input/ng-se_beta_0.0000.asc
   _pre_sim_200k_pop_run/input/ng-se_treatment_seeking_normalized_0.000.asc
   _pre_sim_200k_pop_run/input/ng-se_initialpopulation_200000.asc


In [5]:
# write an index file listing generated items and parameters
binned_population_and_treatment_per_district_data = []
for b in pop_calibration_config["betas"]:
    for a in pop_calibration_config["treatment_access_rates"]:
        for p in pop_calibration_config["population_bins"]:
            beta_str = f"{b:.4f}"
            access_str = f"{a:.3f}"
            population_bin_str = f"{p}"
            binned_population_and_treatment_per_district_data.append({"input_file": f"input_beta_{beta_str}_access_{access_str}_pop_{population_bin_str}.yml", 
                         "beta_raster": b, "access_rate": a, "beta_raster": f"{country_code}_beta_{beta_str}.asc", 
                         "access_raster": f"{country_code}_treatmentseeking_{access_str}.asc",
                         "beta" : f"{b:.4f}", "access_rate": f"{a:.3f}", "population_bin": f"{population_bin_str}"})
index_path = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, "inputs.csv")
pd.DataFrame(binned_population_and_treatment_per_district_data).to_csv(index_path, index=False)
print(f"Wrote index: {index_path}")
        

Wrote index: _pre_sim_200k_pop_run/input/inputs.csv


In [6]:
seasonality_file = f"{template_path}/{country_code}_seasonality.csv"
if SEASONALLITY_MODE == "one":
    if os.path.exists(seasonality_file):
        #Remove _1_pattern_1_location from filename for output
        dest = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, seasonality_file.replace("_1_pattern_1_location", "").replace(template_path + "/", ""))
        with open(seasonality_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        print(f"Copied {seasonality_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {seasonality_file}. Please ensure it is available in the current directory.")
else:     
    seasonality_file = f"{template_path}/seasonality_multiple_patterns_1_location.csv"
    if os.path.exists(seasonality_file):
        #Remove _1_pattern_1_location from filename for output
        dest = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, seasonality_file.replace("_multiple_patterns_1_location", "").replace(template_path + "/", ""))
        with open(seasonality_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        print(f"Copied {seasonality_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {seasonality_file}. Please ensure it is available in the current directory.")

Copied calibration_template_files/ng-se_seasonality.csv to _pre_sim_200k_pop_run/input/ng-se_seasonality.csv


In [7]:
#copy other raster files that are needed for the model but not generated here (e.g. population, districts, traveltime, etc.)
other_files = [
    f"{template_path}/{country_code}_districts_1_location.asc", 
    f"{template_path}/{country_code}_traveltime_1_location.asc",
    # f"{template_path}/{country_code}_districts.asc"
    ]

# Rename if file is seasonality file with different name
for o_file in other_files:
    if os.path.exists(o_file):
        #Remove _1_location from filename for output
        dest = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, o_file.replace("_1_location", "").replace(template_path + "/", ""))
        with open(o_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        print(f"Copied {o_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {o_file}. Please ensure it is available in the current directory.")

Copied calibration_template_files/ng-se_districts_1_location.asc to _pre_sim_200k_pop_run/input/ng-se_districts.asc
Copied calibration_template_files/ng-se_traveltime_1_location.asc to _pre_sim_200k_pop_run/input/ng-se_traveltime.asc


In [8]:
for entry in binned_population_and_treatment_per_district_data:
    input_file_entry       = entry['input_file']
    beta_raster_entry      = entry['beta_raster']
    access_rate_entry      = entry['access_rate']
    access_raster_entry    = entry['access_raster']
    beta_entry             = entry['beta']
    population_bin_entry   = entry['population_bin']

    print(f"input_file:     {input_file_entry}")
    print(f"beta_raster:    {beta_raster_entry}")
    if float(access_rate_entry) == 0:
        ok(f"access_rate:    {access_rate_entry}")
    else:
        warn(f"access_rate:    {access_rate_entry}")
    print(f"access_raster:  {access_raster_entry}")
    if float(beta_entry) == 0:
        ok(f"beta:           {beta_entry}")
    else:
        warn(f"beta:           {beta_entry}")
    print(f"population_bin: {population_bin_entry}")

input_file:     input_beta_0.0000_access_0.000_pop_200000.yml
beta_raster:    ng-se_beta_0.0000.asc
✓ access_rate:    0.000
access_raster:  ng-se_treatmentseeking_0.000.asc
✓ beta:           0.0000
population_bin: 200000


### Run simulation on 200k population with initial population and check sim growth rate agianst target growth rate

Copy country's seasonality_1_location.csv into {calibration_path} directory

In [9]:
percent_error_threshold = 1.0

In [10]:
print(f"Pre calibration run path: {pre_sim_run_path}")
print(f"Birth rate: {birth_rate:.5f}")

Pre calibration run path: _pre_sim_200k_pop_run
Birth rate: 0.03620


In [11]:
bin_path = f"./{pre_sim_run_path}/bin/MalaSim"
# input_path = os.path.join(f"./", PRE_SIM_RUN_PATH_INPUTS_DIR, f"input_beta_0.0000_access_0.000_pop_{population_bin_str}.yml")
# bin_path = f"./{pre_sim_run_path}/bin/MalaSim"
input_path = os.path.join(PRE_SIM_RUN_PATH_INPUTS_DIR, f"input_beta_0.0000_access_0.000_pop_{population_bin_str}.yml")
os.makedirs(pre_sim_analysis_path, exist_ok=True)

In [12]:
log_path = Path(f"{pre_sim_run_path}/log")
output_path = Path(f"{pre_sim_run_path}/output")
script_path = Path(f"{pre_sim_run_path}")

os.makedirs(log_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.makedirs(script_path, exist_ok=True)

cleanup_paths = [log_path, output_path]

for cleanup_path in cleanup_paths:
    clean_directory(cleanup_path)

print("")
info(f"Using population scale: {pre_calibration_sim_scale}")
print("")

print(f"Using bin path: {bin_path}")
print(f"Using input file: {input_path}")
print(f"Using output directory: {output_path}")
print(f"Using analysis directory: {pre_sim_analysis_path}")
print(f"Using calibration path: {calibration_path}\n")

cmd = f"{bin_path} -i {input_path} -o {output_path}/pop_calib_ -r SQLiteMonthlyReporter -v 1"
print(f"Running calibration job with command: {cmd}\n")

info(f"Using birth rate: {birth_rate * 1000} per 1000 births per year")

Number of items in _pre_sim_200k_pop_run/log: 0
Removed 0 items from _pre_sim_200k_pop_run/log
Number of items in _pre_sim_200k_pop_run/output: 1
Removed 1 items from _pre_sim_200k_pop_run/output

→ Using population scale: 0.0025

Using bin path: ./_pre_sim_200k_pop_run/bin/MalaSim
Using input file: _pre_sim_200k_pop_run/input/input_beta_0.0000_access_0.000_pop_200000.yml
Using output directory: _pre_sim_200k_pop_run/output
Using analysis directory: _pre_sim_200k_pop_run/analysis
Using calibration path: calibration_10_one_pattern_25_replicates

Running calibration job with command: ./_pre_sim_200k_pop_run/bin/MalaSim -i _pre_sim_200k_pop_run/input/input_beta_0.0000_access_0.000_pop_200000.yml -o _pre_sim_200k_pop_run/output/pop_calib_ -r SQLiteMonthlyReporter -v 1

→ Using birth rate: 36.2 per 1000 births per year


In [13]:
os.system(cmd)

[2026-08-07 21:43:10] [info] Default logger initialized.
[2026-08-07 21:43:10] [info] Malaria Simulation v6.1.0 (branch: bonilab/malasim-v6-SURF/main, commit: 15744b7)
[2026-08-07 21:43:10] [info] Starting...
[2026-08-07 21:43:10] [info] Parsing command line arguments
[2026-08-07 21:43:10] [info] Verbosity level set to 1. Info, debug will be logged.
[2026-08-07 21:43:10] [info] Loading configuration file: _pre_sim_200k_pop_run/input/input_beta_0.0000_access_0.000_pop_200000.yml
[2026-08-07 21:43:10] [info] Configuration file loaded successfully: _pre_sim_200k_pop_run/input/input_beta_0.0000_access_0.000_pop_200000.yml
[2026-08-07 21:43:10] [info] Using random seed number
[2026-08-07 21:43:10] [info] chromosome_node: 5
[2026-08-07 21:43:10] [info] chromosome_node: 7
[2026-08-07 21:43:10] [info] chromosome_node: 13
[2026-08-07 21:43:10] [info] chromosome_node: 14
[2026-08-07 21:43:10] [info] Found version6_pfpr_incidence_calibrations section — parsing overrides
[2026-08-07 21:43:10] [inf

[2026-08-07 21:43:10] [info] SQLiteDbReporter Creating all reporting tables
[2026-08-07 21:43:10] [info] SQLiteDbReporter Creating reporting tables for level
[2026-08-07 21:43:10] [info] SQLiteDbReporter Creating reporting tables for level
[2026-08-07 21:43:11] [info] SQLiteMonthlyReporter: recorded 10 'version6_pfpr_incidence_calibrations' configuration rows.
[2026-08-07 21:43:11] [info] Model initialized reporters.
[2026-08-07 21:43:11] [info] Model initialized scheduler.
[2026-08-07 21:43:11] [info] Model initialized treatment strategy.
[2026-08-07 21:43:11] [info] Model initialized second-line treatment strategy.
[2026-08-07 21:43:11] [info] Model initialized treatment coverage model.
[2026-08-07 21:43:11] [info] Model initialized data collector.
[2026-08-07 21:43:11] [info] Model initializing population...
[2026-08-07 21:43:11] [info] Model initialized population.
[2026-08-07 21:43:11] [info] Kernel prepared for BurkinaFasoSM, 1 locations, 0.0 MB
[2026-08-07 21:43:11] [info] Prepa

0

In [14]:
initial_observed_population_raster = read_raster(f"{data_path}/{country_code}_initpopulation_{initial_year}.asc")
# target_population_raster = read_raster(f"{data_path}/{input_population_file}")
target_population_raster = read_raster(f"{data_path}/{country_code}_population_projected_{calibration_year}.asc")
# sum non -9999 values in raster to get total population
initial_observed_population = np.nansum(initial_observed_population_raster[0][initial_observed_population_raster[0] != -9999])
target_final_population = np.nansum(target_population_raster[0][target_population_raster[0] != -9999])
print(f"Initial population from raster ({initial_year}): {initial_observed_population:,.0f}")
print(f"Target population from raster ({calibration_year}): {target_final_population:,.0f}")

pop_calib_output_path = os.path.join(output_path, "pop_calib_monthly_data_0.db")
print (f"Reading calibration output from: {pop_calib_output_path}")
population_bin_and_treatment_key_value_pair = get_table(pop_calib_output_path, "monthly_site_data_district")

month_ids = np.array(sorted(population_bin_and_treatment_key_value_pair["monthly_data_id"].unique()))
population_by_month = np.array([
    population_bin_and_treatment_key_value_pair[population_bin_and_treatment_key_value_pair["monthly_data_id"] == month]["population"].sum()
    for month in month_ids
])

starting_pop = population_by_month[0]
last_month = month_ids[-1]
ending_population = population_by_month[-1]

growth_rate = (ending_population / starting_pop) ** (1 / (calibration_year - initial_year)) - 1
backward_growth_rate = 1 / (1 + growth_rate)
inferred_population_initial = target_final_population * (backward_growth_rate ** (calibration_year - initial_year))

percent_error_in_growth_rate = 100 * (growth_rate - target_growth_rate) / target_growth_rate

print(f"Starting population: {starting_pop:,.0f}")
print(f"Ending population: {ending_population:,.0f}\n")

info(f"Inferred initial population: {inferred_population_initial:,.0f}")
info(f"Target final population: {target_final_population:,.0f}\n")

info(f"Birth rate: {birth_rate * 1000} per 1000\n")
print(f"Sim growth rate: {growth_rate:.5f}")
print(f"Target growth rate: {target_growth_rate:.4f}")
if abs(percent_error_in_growth_rate) > percent_error_threshold:
    warn(f"Percent error in growth rate: {percent_error_in_growth_rate:.2f}% exceeds threshold of {percent_error_threshold}%")
else:
    ok(f"Percent error in growth rate: {percent_error_in_growth_rate:.2f}% is within threshold of {percent_error_threshold}%")
print(f"Sim backward growth rate: {backward_growth_rate}")

Initial population from raster (2011): 19,167,271
Target population from raster (2024): 26,716,457
Reading calibration output from: _pre_sim_200k_pop_run/output/pop_calib_monthly_data_0.db


Starting population: 500
Ending population: 651

→ Inferred initial population: 20,519,552
→ Target final population: 26,716,457

→ Birth rate: 36.2 per 1000

Sim growth rate: 0.02051
Target growth rate: 0.0211
⚠ Percent error in growth rate: -2.95% exceeds threshold of 1.0%
Sim backward growth rate: 0.9799045414428798


In [15]:
initial_pop_raster, pop_meta = read_raster(f"{data_path}/{country_code}_initpopulation_{initial_year}.asc")
final_pop_raster, _ = read_raster(f"{data_path}/{country_code}_population_{final_observed_year}.asc")
NODATA = pop_meta["NODATA_value"]
projected_pop_raster, _ = read_raster(f"{data_path}/{country_code}_population_projected_{calibration_year}.asc")
mask = initial_pop_raster != NODATA
total_initial_pop = np.nansum(initial_pop_raster[mask])
total_final_pop = np.nansum(final_pop_raster[mask])
total_projected_pop = np.nansum(projected_pop_raster[mask])
print(f"Total initial observed population ({initial_year} world pop): {total_initial_pop:,.0f}")
print(f"Total final observed population ({final_observed_year} world pop): {total_final_pop:,.0f}")
print(f"Total projected population ({calibration_year}): {total_projected_pop:,.0f}")

Total initial observed population (2011 world pop): 19,167,271
Total final observed population (2020 world pop): 24,572,163
Total projected population (2024): 26,716,457


In [16]:
TARGET_INIT_POP = inferred_population_initial
print(f"Target initial population: {TARGET_INIT_POP:,.0f}")
scale_factor = TARGET_INIT_POP / total_initial_pop
print(f"Scale factor: {scale_factor:.4f}")

initial_pop_raster_inferred = initial_pop_raster.copy().astype(float)
mask = initial_pop_raster_inferred != NODATA
initial_pop_raster_inferred[mask] = initial_pop_raster_inferred[mask] * scale_factor
total_inferred_initial_pop = np.nansum(initial_pop_raster_inferred[mask])
print(f"Total inferred initial population: {total_inferred_initial_pop:,.0f}")

Target initial population: 20,519,552
Scale factor: 1.0706
Total inferred initial population: 20,519,552


Write New Initial Population Raster for Simulation

In [17]:
# nodata_mask = (initial_pop_raster == NODATA)
# initial_pop_raster_inferred = np.round(initial_pop_raster_inferred).astype(np.int32)
# initial_pop_raster_inferred[nodata_mask] = int(NODATA)

# write_raster(
#     initial_pop_raster_inferred,
#     f"{data_path}/{country_code}_initpopulation_{initial_year}_inferred_for_sim.asc",
#     xllcorner=pop_meta["xllcorner"],
#     yllcorner=pop_meta["yllcorner"],
#     cellsize=pop_meta["cellsize"],
#     nodata=int(NODATA),
#     fmt="%d",  # No decimal places for population counts
# )

Check new written initial population raster

In [18]:
initial_pop_raster_test, pop_meta = read_raster(f"{data_path}/{country_code}_initpopulation_{initial_year}_inferred_for_sim.asc")
mask = initial_pop_raster_test != NODATA
total_initial_pop = np.nansum(initial_pop_raster_test[mask])
info(f"Total inferred initial population: \t\t\t\t\t{total_inferred_initial_pop:.0f}")
info(f"Total inferred initial population from newly written raster : \t{total_initial_pop:.0f} (should be very close to above value, only rounding differences)")

→ Total inferred initial population: 					20519552
→ Total inferred initial population from newly written raster : 	20369947 (should be very close to above value, only rounding differences)


### Generate zero beta raster

In [19]:
initial_pop_raster_to_beta, pop_meta = read_raster(f"{data_path}/{country_code}_initpopulation_{initial_year}_inferred_for_sim.asc")
mask = initial_pop_raster_to_beta != NODATA
nodata_mask = (initial_pop_raster_to_beta == NODATA)
beta_zeros = initial_pop_raster.copy().astype(float)
beta_zeros[mask] = 0.0
beta_zeros[nodata_mask] = int(NODATA)

write_raster(
    beta_zeros,
    f"{data_path}/{country_code}_beta_zero.asc",
    xllcorner=pop_meta["xllcorner"],
    yllcorner=pop_meta["yllcorner"],
    cellsize=pop_meta["cellsize"],
    nodata=int(NODATA),
    fmt="%d",  # No decimal places for population counts
)

In [20]:
district_raster, pop_meta = read_raster(f"{data_path}/{country_code}_district.asc")
mask = district_raster != NODATA
district_df = pd.DataFrame({
    "district_id": district_raster[mask].astype(int),
    "population": initial_pop_raster[mask].astype(int),
})
district_pop = district_df.groupby("district_id")["population"].sum().reset_index()
print(district_pop)

   district_id  population
0            7     5042202
1            9     3126634
2           11     2561567
3           26     2170074
4           32     3712017
5           35     2554777


### Remap district raster to sequential IDs starting from 1 if required

In [21]:
def remap_district_ids_sequential(input_district_raster, output_district_raster, output_mapping_csv):
    """
    Remap district IDs in a raster to sequential IDs (1, 2, 3, ...) sorted ascending,
    and save both the remapped raster and the old→new ID mapping as a CSV.
    """
    INPUT_DISTRICT_RASTER = Path(input_district_raster)
    OUTPUT_DISTRICT_RASTER = Path(output_district_raster)
    OUTPUT_MAPPING_CSV = Path(output_mapping_csv)

    # ── 1. Load ──────────────────────────────────────────────────────────────
    district_raster_raw, district_meta = read_raster(INPUT_DISTRICT_RASTER)
    nodata_val = district_meta["NODATA_value"]

    # ── 2. Build mapping: old ID → new sequential ID (sorted ascending) ──────
    valid_ids = np.unique(district_raster_raw[district_raster_raw != nodata_val]).astype(int)
    id_mapping = {old_id: new_id for new_id, old_id in enumerate(np.sort(valid_ids), start=1)}

    mapping_df = pd.DataFrame(
        {"old_district_id": list(id_mapping.keys()),
         "new_district_id": list(id_mapping.values())}
    ).sort_values("new_district_id").reset_index(drop=True)

    print("District ID mapping:")
    print(mapping_df.to_string(index=False))

    # ── 3. Remap raster ────────────────────────────────────────────────────
    district_raster_seq = district_raster_raw.copy()
    for old_id, new_id in id_mapping.items():
        district_raster_seq[district_raster_raw == old_id] = new_id
    # Ensure NODATA cells are preserved
    district_raster_seq[district_raster_raw == nodata_val] = nodata_val

    print(f"\nUnique values in remapped raster (excl. NODATA): "
          f"{sorted(np.unique(district_raster_seq[district_raster_seq != nodata_val]).astype(int))}")

    # ── 4. Write outputs ────────────────────────────────────────────────────
    write_raster(
        district_raster_seq,
        OUTPUT_DISTRICT_RASTER,
        xllcorner=district_meta["xllcorner"],
        yllcorner=district_meta["yllcorner"],
        cellsize=int(district_meta["cellsize"]),
        mask_raster=None,
        nodata=nodata_val,
        fmt="%d",
    )
    mapping_df.to_csv(OUTPUT_MAPPING_CSV, index=False)

    print(f"\nSaved: {OUTPUT_DISTRICT_RASTER}")
    print(f"Saved: {OUTPUT_MAPPING_CSV}")

In [22]:
INPUT_DISTRICT_RASTER  = Path(f"{data_path}/{country_code}_district.asc")
OUTPUT_DISTRICT_RASTER = Path(f"{data_path}/{country_code}_district_seq1.asc")
OUTPUT_MAPPING_CSV  = Path(f"{data_path}/{country_code}_district_seq1_mapping.csv")

Check if the districts are already sequential (e.g. 1,2,3,4,5...) or not (e.g. 6, 9, 13) and create "districts seq1" raster asc and mapping csv if not already sequential

In [23]:
unique_ids = np.unique(district_raster[district_raster != NODATA]).astype(int)
min_id = unique_ids.min() if unique_ids.size > 0 else None
max_id = unique_ids.max() if unique_ids.size > 0 else None
expected_ids = np.arange(1, max_id + 1) if max_id is not None else np.array([])
is_sequential = unique_ids.size > 0 and np.array_equal(unique_ids, expected_ids)
missing_ids = np.setdiff1d(expected_ids, unique_ids)
extra_ids = np.setdiff1d(unique_ids, expected_ids)
print(f"District raster path: {INPUT_DISTRICT_RASTER}")
print(f"Unique district IDs count: {unique_ids.size}")
print(f"Min district ID: {min_id}")
print(f"Max district ID: {max_id}")
print(f"IDs sequential 1..{max_id}: {is_sequential}")
if not is_sequential:
    print(f"Missing IDs: {missing_ids.tolist()}")
    print(f"Extra IDs outside 1..{max_id}: {extra_ids.tolist()}")
    print("Remapping to sequential IDs is needed.")
    remap_district_ids_sequential(INPUT_DISTRICT_RASTER, OUTPUT_DISTRICT_RASTER, OUTPUT_MAPPING_CSV)
    # info(f"BE SURE TO USE THE REMAPPED DISTRICT RASTER IN THE CALIBRATION INPUTS, NOT THE ORIGINAL!")
    ok(f"Use the remapped district raster {OUTPUT_DISTRICT_RASTER}")
    error(f"Do not use {INPUT_DISTRICT_RASTER}!")
else:
    print("District IDs are already sequential and remapping is not needed.")
    # info(f"BE SURE TO USE THE ORIGINAL DISTRICT RASTERIN THE CALIBRATION INPUTS, NOT A REMAPPED VERSION!")
    ok(f"Use the original district raster {INPUT_DISTRICT_RASTER}")
    error(f"Do not use {OUTPUT_DISTRICT_RASTER}!")

District raster path: data/ng-se_district.asc
Unique district IDs count: 6
Min district ID: 7
Max district ID: 35
IDs sequential 1..35: False
Missing IDs: [1, 2, 3, 4, 5, 6, 8, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 33, 34]
Extra IDs outside 1..35: []
Remapping to sequential IDs is needed.
District ID mapping:
 old_district_id  new_district_id
               7                1
               9                2
              11                3
              26                4
              32                5
              35                6

Unique values in remapped raster (excl. NODATA): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]

Saved: data/ng-se_district_seq1.asc
Saved: data/ng-se_district_seq1_mapping.csv
✓ Use the remapped district raster data/ng-se_district_seq1.asc
✗ Do not use data/ng-se_district.asc!


# Validate model growth rate

In [24]:
run_folder = Path(pre_calibration_run_path)
run_folder.mkdir(exist_ok=True)

run_inputs_folder = Path(PRE_CALIBRATION_RUN_INPUTS_DIR)

# delete any yaml files in the run input folder to avoid confusion 
for file in run_inputs_folder.glob("*.yaml"):
    file.unlink()

# copy bin folder from template to pre_calibration path
template_bin = os.path.join(template_path, "bin")
run_bin = os.path.join(run_folder, "bin")

if os.path.exists(template_bin):
    if os.path.exists(run_bin):
        info(f"Bin folder already exists in run path: {run_bin}. Skipping copy.")
    else:
        shutil.copytree(template_bin, run_bin, copy_function=shutil.copy2, symlinks=True)
        ok(f"Copied bin folder from {template_bin} to {run_bin}")

→ Bin folder already exists in run path: _pre_calibration_population_growth_rate_validation/bin. Skipping copy.


Copy input config yml to pre calibration run folder

<span style="color:red">Make sure to use correct district/district_seq1 file version!</span>

In [25]:
TEMPLATE = "input_population_bins.yml"

run_input_dir = Path(run_folder) / "input"
run_input_dir.mkdir(exist_ok=True)

output_dir = Path(run_folder) / "output"
output_dir.mkdir(exist_ok=True)

run_folder_full_path = os.path.abspath(run_folder)
print(f"Run folder full path: {run_folder_full_path}")

with open(TEMPLATE, "r", encoding="utf-8") as f:
    input_template_text = f.read()    

out_text = input_template_text.replace(f"{country_code}_initialpopulation#POPULATION#.asc", input_population_file)
out_text = out_text.replace("#BETA#", "_zero")
out_text = out_text.replace("#ACCESS_RATE#", "")
out_text = out_text.replace("#BIRTH_RATE#", birth_rate_str)
out_text = out_text.replace("#CALIBRATION_YEAR#", f"{calibration_year}")
out_text = out_text.replace("#COUNTRY_CODE#", f"{country_code}")
out_text = out_text.replace("#INITIAL_YEAR#", f"{initial_year}")
out_text = out_text.replace("#INPUT_PATH#", "input")
out_text = out_text.replace("#POPULATION_SCALE#", f"{pre_calibration_population_validation_scale}")
out_text = out_text.replace("initialpopulation#POPULATION#", f"initpopulation_{initial_year}_inferred_for_sim")
out_text = out_text.replace(f"{country_code}_seasonality_1_location", f"{country_code}_seasonality")
out_text = out_text.replace(f"{country_code}_treatmentseeking", f"{country_code}_treatment_seeking_normalized")
out_name = os.path.abspath(os.path.join(run_input_dir, f"input_population_bins_beta_zero_pop_{pre_calibration_population_validation_scale}.yml"))

# out_text = out_text.replace("district.asc", "districts.asc")
out_text = out_text.replace("district.asc", "district_seq1.asc")

with open(out_name, "w", encoding="utf-8") as outf:
    outf.write(out_text)
    print(f"Created input file: {out_name}\n")
    
# Copy other raster files that are needed for the model but not generated here (e.g. population, districts, traveltime, etc.)
other_files = [
    f"{data_path}/{country_code}_beta_zero.asc",
    # f"{calibration_analysis_path}/{country_code}_beta.asc",
    f"{data_path}/{country_code}_initpopulation_{initial_year}_inferred_for_sim.asc", 
    # f"{data_path}/{country_code}_districts.asc", 
    f"{data_path}/{country_code}_district_seq1.asc", 
    f"{data_path}/{country_code}_traveltime.asc",
    # f"{data_path}/{country_code}_treatmentseeking.asc",
    f"{data_path}/{country_code}_treatment_seeking_normalized.asc",
    ]
for o_file in other_files:
    if os.path.exists(o_file):
        # print(f"\nCopying {o_file} to {VALIDATION_RUN_INPUTS_DIR}")
        dest = os.path.abspath(os.path.join(run_input_dir, o_file.replace(f"{data_path}/", "").replace(f"{calibration_analysis_path}/", "")))
        with open(o_file, "r", encoding="utf-8") as srcf:
            with open(dest, "w", encoding="utf-8") as destf:
                destf.write(srcf.read())
        ok(f"Copied {o_file} to {dest}")
    else:
        warn(f"Warning: required raster not found: {o_file}. Please ensure it is available in the current directory.") 

Run folder full path: /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation
Created input file: /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/input_population_bins_beta_zero_pop_0.25.yml

✓ Copied data/ng-se_beta_zero.asc to /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/ng-se_beta_zero.asc
✓ Copied data/ng-se_initpopulation_2011_inferred_for_sim.asc to /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/ng-se_initpopulation_2011_inferred_for_sim.asc
✓ Copied data/ng-se_district_seq1.asc to /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/ng-se_district_seq1.asc


✓ Copied data/ng-se_traveltime.asc to /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/ng-se_traveltime.asc
✓ Copied data/ng-se_treatment_seeking_normalized.asc to /work/tuv89272/Calibration_Pipeline_v6_NG-SE/_pre_calibration_population_growth_rate_validation/input/ng-se_treatment_seeking_normalized.asc


### Create run.sh script

In [26]:
# clean 0.log, malaSim_0.pid, and run.sh files if they exist
for file_name in ["0.log", "malaSim_0.pid", "run.sh"]:
    file_path = run_folder / file_name
    if file_path.exists():
        file_path.unlink()
        print(f"Removed existing file: {file_path}")

Removed existing file: _pre_calibration_population_growth_rate_validation/run.sh


In [27]:
run_sh_code = f"""#!/bin/bash

cd {run_folder_full_path} || exit 1

nohup ./bin/MalaSim \\
-i input/input_population_bins_beta_zero_pop_{pre_calibration_population_validation_scale}.yml \\
-r SQLiteMonthlyReporter \\
-o output/pop_validation \\
-j 0 \\
-v 1 \\
> 0.log 2>&1 &

echo $! > malaSim_0.pid
echo "MalaSim started in background"
echo "PID: $(cat malaSim_0.pid)"
echo "Log: 0.log"
"""

run_sh_path = run_folder / "run.sh"
run_sh_path.write_text(run_sh_code)
run_sh_path.chmod(0o755)  # make it executable

info(f"Written to: {run_sh_path} with scale {pre_calibration_population_validation_scale}")

→ Written to: _pre_calibration_population_growth_rate_validation/run.sh with scale 0.25


Run <code>./run.sh</code> in terminal